# GeoFire-X — unique Focal, préparation finale après PASS
Le PASS Kaggle valide environnement, initialisation et corpus. Aucun entraînement effectué.
Paquet scientifique et configuration inchangés ; seule la logistique du notebook évolue.
Internet OFF. Attacher les 11 datasets privés EXISTANTS, aucune réimportation/publication.
GPU T4 x2 peut être proposé par Kaggle ; CUDA_VISIBLE_DEVICES=0 expose UNE seule T4.
Ne pas lancer Run All/Save Version avec exécution avant autorisation explicite.
Réserve de session proposée : 120 min, dont 45 min maximum de fit/session (pause existante).
Contrôle : 12 GiB libres sur working et tmp ; AP exacte non approximée.
RUN_TRAINING=False ; aucun fichier d'autorisation active n'est livré.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"  # BEFORE torch/CUDA import
import base64, hashlib, io, json, os, sys, tempfile, zipfile, subprocess
from pathlib import Path
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='8f48deba428b11be05cc20ccaaf9c4ef3dec19445a4479ccfbf9dd19355a16bd'
workspace=Path(tempfile.mkdtemp(prefix="geofirex-focal-code-",dir="/tmp"))
with zipfile.ZipFile(io.BytesIO(payload)) as z: z.extractall(workspace)
code_archive=workspace/"focal-code.zip"
code_archive.write_bytes(payload)
sys.path.insert(0,str(workspace))
os.environ["PYTHONPATH"]=str(workspace)+os.pathsep+os.environ.get("PYTHONPATH", "")

extra=base64.b64decode('...')
assert hashlib.sha256(extra).hexdigest()=='71f8cf79764bef15f13cc99951b540ba686c7f5fbc77881126d505ea7b846f34'
with (workspace/'focal_run_logistics.py').open("xb") as f: f.write(extra)

extra=base64.b64decode('...')
assert hashlib.sha256(extra).hexdigest()=='d52dccc18e6caa71532c6c73de100d7294aff87d5d3a4fb785f401459a3ac3af'
with (workspace/'precheck_report_kaggle_PASS.json').open("xb") as f: f.write(extra)


In [ ]:
# Read-only byte verification; NO NPZ decode, model or GPU query.
from geofirex.calibration_mounts import restore_verified
expected=['ed88f93ae8329c57f34709fdf3a59e3ec3272db8238f843d4329f2cb23e293d0', '4aa8b2818192a2cc67941f1c1bbb8a1001e144114486bb05ecaa9b504993c6fa', 'b5caf48618be26a75a480d91401f9de0d0936607936d09556f6308c1d76c5dcd', 'ba72fbbc58f398a7bcd471d5cb21aebe4d11ed68da02caa912c2e746ccadc04d', 'ada2f8b9a3f7f340efed521b15bf3fe4e3b4151fa39eab06ca2055518f58ab58', '3bcde37b262d5828f98153350d961e0ad771032fdedb04ff180ace536e4cb33a', '001a171f309ac58f4160269d46b9163b0c0c26c7a488cc545f6fa1b42e55d72a', '906b02993f756f4b6098ac8da11ecac173f59d20ebda70c8d2bd3bb37b8f6dbd', '793223fda8a5f0d06e792bcd09b2604a97252952a5323b144184a8e4f1496a4f', 'af3015768e0174071c4d197ad31681ef20fbe0cc1de2c1084254bfef5b91393f', '49e5551860a6d14c020c86dc33118a7cd2e0ca3dc9b470e6e95af6061817fa81']
maps=[p for p in Path("/kaggle/input").rglob("mount-map.json")
      if json.loads(p.read_text())["identity"] in expected]
corpus=restore_verified(maps,expected,Path(tempfile.mkdtemp(prefix="geofirex-focal-mount-",dir="/tmp"))/"corpus")
command=[sys.executable,"-m","geofirex.focal_experiment","--config",str(workspace/"focal_config.json"),
         "--corpus",str(corpus),"--index",str(corpus/"index")]
subprocess.run(command,check=True,env=os.environ.copy())

import focal_run_logistics as logistics
logistics.identity(workspace)
print("Disk free /working:", logistics.disk_check("/kaggle/working"))
print("Disk free /tmp:", logistics.disk_check("/tmp"))
print("PASS accepted; frozen science unchanged; training remains disabled.")

import torch, numpy as np
assert torch.__version__=="2.11.0+cu130" and np.__version__=="2.1.3", "Environment drift; do not train"


## Autorisation explicite requise
Conserver RUN_TRAINING=False jusqu'à l'accord explicite portant sur UN run et le budget complet.
Ensuite seulement créer le fichier externe selon authorization_TEMPLATE_DISABLED.json :
hashes config/code/logistique, runs=1, max_training_seconds=2700, session_reserve_minutes=120,
gpu_count=1, destinations locale/externe et authorized_by. Ce modèle n'autorise rien.
Le verrou de session interdit un second lancement. Aucun resume/retrain automatique.


In [ ]:
RUN_TRAINING=False
AUTHORIZATION_FILE=None
OUT=None
if RUN_TRAINING:
    if AUTHORIZATION_FILE is None: raise RuntimeError("Explicit authorization required")
    import uuid
    OUT=Path("/kaggle/working")/("geofirex-focal-"+uuid.uuid4().hex)
    execution=[sys.executable,str(workspace/"focal_run_logistics.py"),"train",
               "--workspace",str(workspace),"--corpus",str(corpus),"--out",str(OUT),
               "--authorization",str(AUTHORIZATION_FILE)]
    subprocess.run(execution,check=True,env=os.environ.copy())
else:
    print("DISABLED: no training/evaluation/GPU call.")


## Sauvegarde et récupération
TRAINING-SAFE.zip + SHA est écrit et relu AVANT l'évaluation32/AP exacte inchangée.
COMPLETE.zip contient sélection32, comparaison historique, métriques, historique et checkpoint.
PARTIAL.zip est tenté en finally en cas d'erreur/pause ; un arrêt brutal du noyau ne peut garantir finally.
Télécharger ZIP + SHA, garder les outputs Kaggle puis exécuter le script backup local/externe.
Ne jamais clôturer la session avant le téléchargement et sa vérification.
Si AP échoue après 23404 updates, utiliser evaluate-only sur CE checkpoint, jamais retrain.
Le guide KAGGLE_CELLULES_ET_SAUVEGARDE.md donne les commandes exactes.


In [ ]:
if RUN_TRAINING and OUT is not None:
    from IPython.display import display, FileLink
    archives=sorted(OUT.parent.glob(OUT.name+"-*.zip"))
    for archive in archives:
        logistics.verify_zip(archive,archive.with_suffix(".sha256").read_text().split()[0])
        display(FileLink(str(archive)))
        display(FileLink(str(archive.with_suffix(".sha256"))))
    if not archives: raise RuntimeError("No verified archive: preserve working directory manually")
